# 20-epoch Qwen3 eight-block learned layer merging (T4 ×2)

**Controlled comparison with the earlier ten-epoch fast-SFT experiment.** Only the merger's training epochs increase to 20; the same seven-language token counts, merger architecture, 13% CE limit, SFT dataset size (200/language, one epoch), and 30-question/language benchmarks are retained. The run uses a distinct repository directory and output directory, keeping all previous files safe. New source uses process-isolated benchmark workers. Enable Kaggle T4 ×2 and Internet. The 3-hour planning budget is advisory; no runtime or step cap stops the 20 epochs.

Do not point the new notebook at an old 10-epoch output directory; it cannot safely resume old implementation/configuration locks.

In [ ]:
from pathlib import Path
import subprocess,sys,os,json,importlib.metadata
ROOT=Path('/kaggle/working');REPO=ROOT/'Layer-Merging-20Epochs';OUTPUT=ROOT/'outputs/qwen3_8block_sft_20epochs'
if not REPO.exists():subprocess.run(['git','clone','https://github.com/amritosanto/Layer-Merging.git',str(REPO)],check=True)
else:subprocess.run(['git','pull','--ff-only'],cwd=REPO,check=True)
assert 'benchmark-worker' in (REPO/'run.py').read_text(), 'Repo does not contain the GPU isolation fix'
print('Checked out:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip())

In [ ]:
packages={'transformers':'4.51.3','peft':'0.15.2','datasets':'4.4.1','accelerate':'1.6.0','bitsandbytes':'0.48.2','huggingface-hub':'0.36.0','tokenizers':'0.21.4'}
missing=[]
for name,version in packages.items():
    try:current=importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:current=None
    if current!=version:missing.append(f'{name}=={version}')
if missing:subprocess.run([sys.executable,'-m','pip','install','-q',*missing],check=True)
print('Retained existing CUDA PyTorch; compatible dependencies installed')

In [ ]:
import torch
assert torch.cuda.device_count()>=2,'Enable Kaggle T4 x2'
print([torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
config=json.loads((REPO/'config.smoke_sft20.json').read_text());config['output_dir']=str(OUTPUT)
assert config['merger_blocks']==8 and config['sft_epochs']==1
assert config['pilot_questions']==30 and config['max_relative_ce_increase']==0.13
assert config['train_tokens_per_language']==10000
assert config['epochs']==20 and config['max_optimizer_steps']==0 and config['training_hours']==0
assert config['train_tokens_per_language']==10000 and config['pilot_questions']==30
assert config['max_relative_ce_increase']==0.13 and config['sft_examples_per_language']==200
OUTPUT.mkdir(parents=True,exist_ok=True)
CONFIG=ROOT/'qwen-smoke-run-fixed.json'
CONFIG.write_text(json.dumps(config,indent=2))
ENV=os.environ.copy();ENV.update(USE_TF='0',USE_FLAX='0',TOKENIZERS_PARALLELISM='false')
def run_stage(stage):
    subprocess.run([sys.executable,'run.py',stage,'--config',str(CONFIG)],cwd=REPO,env=ENV,check=True)

## Prepare datasets and GPU preflight

In [ ]:
run_stage('prepare')
run_stage('preflight')

## Train merger, search feasible compression, and export NF4

These stages may be long. Their outputs are resumable.

In [ ]:
run_stage('train')

In [ ]:
run_stage('search')
run_stage('export')

## Audited SFT and two-GPU training

In [ ]:
run_stage('sft-prepare')

In [ ]:
run_stage('sft-both')

In [ ]:
run_stage('sft-verify')

## Corrected benchmark and report

Each model is loaded in one isolated worker **per GPU**. The two GPU workers run concurrently; existing completed predictions can be reused.

In [ ]:
run_stage('benchmark')

In [ ]:
run_stage('report')
from IPython.display import display,FileLink
display(FileLink(str(OUTPUT/'REPORT.md')))

## Archive all Kaggle working files

Safe to run separately if a preceding stage fails. Only archive completed reports unless salvaging.

In [ ]:
import zipfile,shutil
from IPython.display import display,FileLink
archive=ROOT/'Qwen3_8Block_20Epoch_SFT_WORKING.zip'
assert (OUTPUT/'REPORT.md').is_file(),'Complete reporting before final archive, or salvage manually'
files=[p for p in ROOT.rglob('*') if p.is_file() and p.resolve()!=archive.resolve()]
required=sum(p.stat().st_size for p in files)
free=shutil.disk_usage(ROOT).free
assert free+(archive.stat().st_size if archive.exists() else 0)>required*1.05,'Insufficient free disk for ZIP'
already={'.pt','.pth','.safetensors','.zip','.gz','.parquet','.arrow','.png','.npz'}
with zipfile.ZipFile(archive,'w',allowZip64=True) as z:
    for p in files:z.write(p,p.relative_to(ROOT),compress_type=zipfile.ZIP_STORED if p.suffix.lower() in already else zipfile.ZIP_DEFLATED)
print('Archive:',archive,'GiB:',round(archive.stat().st_size/(2**30),2))
display(FileLink(str(archive)))